In [1]:
import numpy as np
import os
import pandas as pd
import re
from pathlib import Path
import pyreadr

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [2]:
DATA_DIR = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")

In [3]:
# ---- Load pre-built metadata instead of rebuilding ----
meta_all = pd.read_csv(DATA_DIR / "phase2_samples_meta.csv", parse_dates=["event_datetime", "t0_datetime", "target_datetime"])
print("Loaded meta_all:", meta_all.shape)
print("Unique events:", meta_all["event_id"].nunique())

# ---- Load arrays (once you save them) ----
data = np.load(DATA_DIR / "phase2_samples.npz")
X_all = data["X"]  # (8602, 3, 501, 371) float32
Y_all = data["Y"]  # (8602, 1, 501, 371) float32
print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)

Loaded meta_all: (8602, 8)
Unique events: 253
X_all: (8602, 3, 501, 371)
Y_all: (8602, 1, 501, 371)


In [4]:
# derive stats ----
Y_flat = Y_all.reshape(Y_all.shape[0], -1)

meta_all["mean_intensity"] = Y_flat.mean(axis=1)
meta_all["max_intensity"]  = Y_flat.max(axis=1)

wet_mask = Y_flat > 0.1
meta_all["mean_intensity_wet"] = [
    Y_flat[i][wet_mask[i]].mean() if wet_mask[i].any() else 0.0
    for i in range(Y_flat.shape[0])
]
meta_all["wet_fraction"] = wet_mask.sum(axis=1) / Y_flat.shape[1]

# ---- build split ----
events = (
    meta_all[["event_id", "event_datetime", "season"]]
    .drop_duplicates()
    .sort_values(["season", "event_datetime"])
)


In [5]:
### Define operational split ###

train_events, val_events, test_events = [], [], []
for season, df in events.groupby("season"):
    df = df.sort_values("event_datetime")
    n = len(df)
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)
    train_events.extend(df.iloc[:train_end]["event_id"])
    val_events.extend(df.iloc[train_end:val_end]["event_id"])
    test_events.extend(df.iloc[val_end:]["event_id"])

meta_all["split"] = "train"
meta_all.loc[meta_all["event_id"].isin(val_events),  "split"] = "val"
meta_all.loc[meta_all["event_id"].isin(test_events), "split"] = "test"

print(meta_all["split"].value_counts())

train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

split
train    5984
test     1360
val      1258
Name: count, dtype: int64


In [ ]:
# ---- Log transform ----
X_all_log = np.log1p(X_all).astype(np.float32)
Y_all_log = np.log1p(Y_all).astype(np.float32)

# ---- Final splits ----
train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

X_train, Y_train = X_all_log[train_mask], Y_all_log[train_mask]
X_val,   Y_val   = X_all_log[val_mask],   Y_all_log[val_mask]
X_test,  Y_test  = X_all_log[test_mask],  Y_all_log[test_mask]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_val   = meta_all.loc[val_mask].reset_index(drop=True)
meta_test  = meta_all.loc[test_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape}  Y_val:   {Y_val.shape}")
print(f"X_test:  {X_test.shape}  Y_test:  {Y_test.shape}")


In [6]:
# ---- Persistence baseline (in original space) ----
Y_persist = X_all[test_mask][:, -1:, :, :]   # shape (N_test, 1, H, W)
mse_persist = float(np.mean((Y_persist - Y_all[test_mask])**2))
mae_persist = float(np.mean(np.abs(Y_persist - Y_all[test_mask])))
print(f"Persistence  MSE: {mse_persist:.6f}  MAE: {mae_persist:.6f}")


Persistence  MSE: 0.237859  MAE: 0.082429


In [7]:
# ---- Persistence CSI @ multiple thresholds ----
Y_true    = Y_all[test_mask]   # (N, 1, H, W) original space
thresholds = [0.1, 0.5, 1.0]
csi_results = {}

for thr in thresholds:
    tp  = np.sum((Y_persist >= thr) & (Y_true >= thr))
    fp  = np.sum((Y_persist >= thr) & (Y_true <  thr))
    fn  = np.sum((Y_persist <  thr) & (Y_true >= thr))
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float("nan")
    csi_results[thr] = csi
    print(f"Persistence  CSI @{thr}mm/10min: {csi:.4f}")



Persistence  CSI @0.1mm/10min: 0.4721
Persistence  CSI @0.5mm/10min: 0.2412
Persistence  CSI @1.0mm/10min: 0.1254


In [8]:
from scipy.ndimage import uniform_filter

def fss_score(pred, obs, threshold, scale_px):
    pred_bin  = (pred >= threshold).astype(np.float32)
    obs_bin   = (obs  >= threshold).astype(np.float32)
    size      = 2 * scale_px + 1
    # box-average of binary field = fraction of rainy pixels in each neighbourhood window
    pred_frac = uniform_filter(pred_bin, size=size, mode="constant")
    obs_frac  = uniform_filter(obs_bin,  size=size, mode="constant")
    fbs       = np.mean((pred_frac - obs_frac) ** 2)
    fbs_worst = np.mean(pred_frac ** 2) + np.mean(obs_frac ** 2)
    return 1.0 - fbs / fbs_worst if fbs_worst > 0 else float("nan")

FSS_THRESHOLDS = [0.1, 0.5, 1.0]
FSS_SCALES_PX  = [8, 32]

# Persistence: last input frame as forecast (original space)
Y_true    = Y_all[test_mask]     # (N, 1, H, W)
Y_persist = X_all[test_mask][:, -1:, :, :]

rows = (
    [{"model": "persistence", "metric": "mse",  "threshold_mm": None, "scale_px": None, "value": mse_persist},
     {"model": "persistence", "metric": "mae",  "threshold_mm": None, "scale_px": None, "value": mae_persist}]
  + [{"model": "persistence", "metric": "csi",  "threshold_mm": thr,  "scale_px": None, "value": csi_results[thr]}
     for thr in [0.1, 0.5, 1.0]]
  + [{"model": "persistence", "metric": "fss",  "threshold_mm": thr,  "scale_px": s,
      "value": np.nanmean([fss_score(Y_persist[i, 0], Y_true[i, 0], thr, s)
                           for i in range(len(Y_true))])}
     for thr in FSS_THRESHOLDS for s in FSS_SCALES_PX]
)

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))
df_metrics.to_csv(DATA_DIR / "persistence_metrics.csv", index=False)


      model metric  threshold_mm  scale_px    value
persistence    mse           NaN       NaN 0.237859
persistence    mae           NaN       NaN 0.082429
persistence    csi           0.1       NaN 0.472064
persistence    csi           0.5       NaN 0.241232
persistence    csi           1.0       NaN 0.125424
persistence    fss           0.1       8.0 0.754800
persistence    fss           0.1      32.0 0.900074
persistence    fss           0.5       8.0 0.521307
persistence    fss           0.5      32.0 0.764242
persistence    fss           1.0       8.0 0.346952
persistence    fss           1.0      32.0 0.612143


In [11]:
### Save train/val/test split ###
meta_all.to_csv(DATA_DIR / "phase2_samples_meta_enriched.csv", index=False)
print("Saved enriched metadata with split column.")

# Then in training you just do:
# meta = pd.read_csv("phase2_samples_meta_enriched.csv")
# train_mask = meta["split"] == "train"
# X_train = X_all_log[train_mask]  # fast, no extra disk space

Saved enriched metadata with split column.
